## Example 1: Basic async MongoDB client setup

In [ ]:
# Example 1: Basic async MongoDB client setup
import motor.motor_asyncio

print("Example 1 - Async MongoDB client")

# Create async MongoDB client
client = motor.motor_asyncio.AsyncIOMotorClient("mongodb://localhost:27017/")
print(f"Client type: {type(client).__name__}")

# Access database and collection
db = client["test_db"]
collection = db["users"]

print(f"Database: {db.name}")
print(f"Collection: {collection.name}")

# Show client attributes
print(f"Has async methods: {hasattr(collection, 'insert_one')}")
print(f"Max pool size: {client.options.pool_options.max_pool_size}")
# Creating the client and these handles does not contact a server
assert isinstance(client, motor.motor_asyncio.AsyncIOMotorClient)
assert db.name == "test_db" and collection.name == "users" and collection.full_name == "test_db.users"
assert client.options.pool_options.max_pool_size == 100  # pymongo's default maxPoolSize

# Client configuration
print("Features: Connection pooling, Replica sets, SSL/TLS, Auth")

client.close()
print("Example 1 completed")

## Example 2: Async CRUD operation methods

In [ ]:
# Example 2: Async CRUD operation methods
import motor.motor_asyncio

print("Example 2 - CRUD operations API")

client = motor.motor_asyncio.AsyncIOMotorClient("mongodb://localhost:27017/")
collection = client["mydb"]["users"]

# Available async methods
crud_methods = [
    "insert_one", "insert_many", "find_one", "find",
    "update_one", "update_many", "delete_one", "delete_many",
    "count_documents", "replace_one"
]

print("CRUD methods available:")
for method in crud_methods:
    has_method = hasattr(collection, method)
    print(f"  {method}: {'✓' if has_method else '✗'}")
assert all(hasattr(collection, m) for m in crud_methods)
assert collection.full_name == "mydb.users"

client.close()
print("Example 2 completed")

## Example 3: Aggregation pipeline API

In [ ]:
# Example 3: Aggregation pipeline API
import motor.motor_asyncio

print("Example 3 - Aggregation pipeline")

client = motor.motor_asyncio.AsyncIOMotorClient("mongodb://localhost:27017/")
collection = client["mydb"]["sales"]

# Example aggregation pipeline
pipeline = [
    {"$match": {"status": "completed"}},
    {"$group": {"_id": "$category", "total": {"$sum": "$amount"}}},
    {"$sort": {"total": -1}},
    {"$limit": 10}
]

print("Aggregation pipeline stages:")
for i, stage in enumerate(pipeline, 1):
    operation = list(stage.keys())[0]
    print(f"  Stage {i}: {operation}")

assert [next(iter(stage)) for stage in pipeline] == ["$match", "$group", "$sort", "$limit"]

# aggregate() returns a lazy async cursor; the pipeline runs on the server only when it is iterated
agg_cursor = collection.aggregate(pipeline)
print(f"aggregate() returns: {type(agg_cursor).__name__}")
assert hasattr(agg_cursor, "to_list")

client.close()
print("Example 3 completed")

## Example 4: GridFS for file storage

In [ ]:
# Example 4: GridFS for file storage
import motor.motor_asyncio

print("Example 4 - GridFS async file storage")

client = motor.motor_asyncio.AsyncIOMotorClient("mongodb://localhost:27017/")
db = client["mydb"]

# Create GridFS instance
fs = motor.motor_asyncio.AsyncIOMotorGridFSBucket(db)
print(f"GridFS type: {type(fs).__name__}")

# GridFS async methods for file storage
gridfs_methods = [
    "open_upload_stream", "open_download_stream",
    "upload_from_stream", "download_to_stream",
    "delete", "find", "rename"
]

print("GridFS async methods:")
for method in gridfs_methods:
    has_method = hasattr(fs, method)
    print(f"  {method}: {'✓' if has_method else '✗'}")
assert isinstance(fs, motor.motor_asyncio.AsyncIOMotorGridFSBucket)
assert all(hasattr(fs, m) for m in gridfs_methods)

print("Use case: Store large files in MongoDB")

client.close()
print("Example 4 completed")

## Example 5: Async cursor iteration

In [ ]:
# Example 5: Async cursor iteration
import motor.motor_asyncio

print("Example 5 - Async cursor operations")

client = motor.motor_asyncio.AsyncIOMotorClient("mongodb://localhost:27017/")
collection = client["mydb"]["products"]

# Create cursor (doesn't execute until iterated)
cursor = collection.find({"price": {"$gt": 100}})
print(f"Cursor type: {type(cursor).__name__}")

# Cursor methods
cursor_methods = [
    "sort", "limit", "skip", "batch_size",
    "to_list", "distinct", "explain"
]

print("Cursor methods:")
for method in cursor_methods:
    has_method = hasattr(cursor, method)
    print(f"  {method}: {'✓' if has_method else '✗'}")
assert isinstance(cursor, motor.motor_asyncio.AsyncIOMotorCursor)
assert all(hasattr(cursor, m) for m in cursor_methods)
# Cursor modifiers chain and return the same cursor without contacting the server
assert cursor.sort("price", -1).limit(5).skip(1) is cursor

print("Usage: async for doc in cursor: ...")

client.close()
print("Example 5 completed")